# Linear Search

In [ ]:

def linear_search_first(arr, key):
    for index, value in enumerate(arr):
        if value == key:
            return index
    return -1


arr = [2, 4, 7, 4, 9, 4, 5]
key = 4
result = linear_search_first(arr, key)
print(result)

The simplest way to find something: check every element one by one, left to right, and stop as soon as you find a match. No assumptions about the data being sorted — it works on anything, but in the worst case it has to look at every item, so it's O(n).

In [ ]:

def search_all_occurrences(arr, key):
    indices = []
    for index, value in enumerate(arr):
        if value == key:
            indices.append(index)
    return indices


arr2 = [3, 4, 5, 6, 6, 5, 4, 3, 3, 2, 1, 3, 4]
key2 = 3
result2 = search_all_occurrences(arr2, key2)
print(result2)

Same walk as before, but instead of stopping at the first match, it keeps going and collects every index where the value shows up.

In [ ]:

class LinearSearch:
    def __init__(self, arr, key, mode="first"):
        self.arr = arr
        self.key = key

        if mode == "first":
            self.result = self.first_occurrence()
        elif mode == "all":
            self.result = self.all_occurrences()
        else:
            raise ValueError("Mode must be 'first' or 'all'")

    def first_occurrence(self):
        for index, value in enumerate(self.arr):
            if value == self.key:
                return index
        return -1

    def all_occurrences(self):
        indices = []
        for index, value in enumerate(self.arr):
            if value == self.key:
                indices.append(index)
        return indices


search1 = LinearSearch(arr, key=4, mode="first")
print("First occurrence of 4:", search1.result)

search2 = LinearSearch(arr2, key=3, mode="all")
print("All occurrences of 3:", search2.result)

This wraps the same two searches into one reusable class instead of two separate functions — you hand it an array, a key, and a `mode`, and it runs the matching search automatically as soon as it's created. Functionally identical to the two functions above, just packaged more conveniently for reuse.

# Binary Search

In [ ]:

def binary_search(arr, target):
    low = 0
    high = len(arr) - 1

    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


sorted_array = [1, 2, 3, 4, 5, 6, 7, 8, 9, 44, 55, 66, 77, 88]
target = 5
result = binary_search(sorted_array, target)
print(result)

Binary search only works on **sorted** data, but in exchange it's much faster than linear search. Instead of checking every element, it looks at the middle one and uses the fact that the array is sorted to immediately rule out half the remaining elements: too small → search the right half, too big → search the left half. Repeating this keeps cutting the search space in half, giving O(log n) instead of O(n).

# Hash Tables

In [ ]:

class LinearProbingTable:
    def __init__(self, size):
        self.size = size
        self.table = [None] * size

    def insert(self, key):
        index = key % self.size
        for i in range(self.size):
            probe_idx = (index + i) % self.size
            if self.table[probe_idx] is None:
                self.table[probe_idx] = key
                return True
        return False

    def search(self, key):
        index = key % self.size
        for i in range(self.size):
            probe_idx = (index + i) % self.size
            if self.table[probe_idx] == key:
                return probe_idx
            if self.table[probe_idx] is None:
                break
        return -1


lp_table = LinearProbingTable(10)
lp_table.insert(12)
lp_table.insert(22)
print(f"Linear Probing: 22 found at index {lp_table.search(22)}")

A hash table stores things at an index computed from the key (`key % size` here), which makes lookups fast — but two different keys can land on the same index. That's a **collision**. Linear probing handles it by simply trying the next slot, then the next, then the next, wrapping around the table, until it finds an empty one. Searching follows the exact same path: keep checking slots until you find the key or hit an empty slot (which means it was never inserted).

In [ ]:

class DoubleHashingTable:
    def __init__(self, size):
        self.size = size
        self.table = [None] * size

    def _hash2(self, key):
        return 7 - (key % 7)

    def insert(self, key):
        h1 = key % self.size
        step = self._hash2(key)
        for i in range(self.size):
            idx = (h1 + i * step) % self.size
            if self.table[idx] is None:
                self.table[idx] = key
                return True
        return False

    def search(self, key):
        h1 = key % self.size
        step = self._hash2(key)
        for i in range(self.size):
            idx = (h1 + i * step) % self.size
            if self.table[idx] == key:
                return idx
            if self.table[idx] is None:
                break
        return -1


dh_table = DoubleHashingTable(10)
dh_table.insert(10)
dh_table.insert(20)
print(f"Double Hashing: 20 found at index {dh_table.search(20)}")

Double hashing is a smarter version of the same idea. Instead of always stepping to the very next slot on a collision (which can cause long clumps of full slots), it uses a **second hash function** to decide how big a jump to take each time. Different keys usually get different step sizes, so collisions spread out across the table more evenly instead of clustering together.

In [ ]:

class SeparateChainingTable:
    def __init__(self, size):
        self.size = size
        self.table = [[] for _ in range(size)]

    def insert(self, key):
        index = key % self.size
        if key not in self.table[index]:
            self.table[index].append(key)

    def search(self, key):
        index = key % self.size
        return key in self.table[index]


sc_table = SeparateChainingTable(5)
sc_table.insert(10)
sc_table.insert(15)
print(f"Separate Chaining: 15 found? {sc_table.search(15)}")

Separate chaining takes a completely different approach to collisions: instead of finding another slot, just let each slot hold a small **list** of everything that hashes there. `10` and `15` both hash to index `0` in a table of size 5, so they simply sit together in that slot's list, and searching just checks whether the key is in that list.